# Teacher Chain-of-Thought Distillation — NVIDIA Nemotron Reasoning Challenge (final leaderboard 0.83)

This notebook trains a **rank-32 LoRA adapter** on `Nemotron-3-Nano-30B-A3B-BF16` to solve in-context rule-discovery puzzles, by distilling **natural chain-of-thought from strong teacher models** that solved the real competition puzzles.

The one idea that makes it work: **answer-consistency cleaning** — a teacher trace is kept only when its final `\boxed{}` answer matches ground truth (with a one-line rounding note for numeric near-misses, and genuine errors dropped). The model learns the teachers' fluent reasoning rather than a rigid template, which is why this is the strongest of the three approaches in this repo.

See the repository README for the competition rules and a comparison of all three notebooks.

## 0 · Load the data

Load the competition `train.csv` (used only to hold out a small evaluation set) and the teacher chain-of-thought dataset that we actually train on.

In [ ]:
# Load the competition data + the teacher chain-of-thought dataset.
# train.csv is used only to carve out a held-out eval set; the teacher CoT is the training signal.

import os
import polars as pl

TRAIN_CANDIDATES = [
    "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv",
    "train.csv",
]
TRAIN_CSV = next((p for p in TRAIN_CANDIDATES if os.path.exists(p)), TRAIN_CANDIDATES[-1])
train = pl.read_csv(TRAIN_CSV)
print(f"Loaded {train.height} train rows from {TRAIN_CSV}")

COT_CANDIDATES = [
    "/kaggle/input/datasets/dgxchen/nemotron-cot-tong/problem_ids_matched.csv",
    "/kaggle/input/nemotron-cot-tong/problem_ids_matched.csv",
    "winner_dataset/problem_ids_matched.csv",
]
COT_CSV = next((p for p in COT_CANDIDATES if os.path.exists(p)), COT_CANDIDATES[0])
assert os.path.exists(COT_CSV), f"teacher CoT not found; add dgxchen/nemotron-cot-tong. Tried {COT_CANDIDATES}"
print("Teacher CoT path:", COT_CSV)


## 1 · Load the base model and attach a LoRA adapter

Load the 30B base in bf16 and attach a rank-32 LoRA on the attention + MLP projections. The base weights stay frozen — only the adapter trains.

In [ ]:
# Load Nemotron-3-Nano-30B in bf16 (no quantization) and attach a rank-32 LoRA on the
# attention + MLP projection layers. The base model is frozen; only the adapter is trained.

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from unsloth import FastLanguageModel
import torch
import kagglehub
import mamba_ssm

assert mamba_ssm is not None

MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
OUTPUT_DIR = "/kaggle/working"
BASE_MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
LORA_RANK = 32
MAX_LENGTH = 8192
MODEL_MAX_SEQ = 16384

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH,
    max_seq_length=MODEL_MAX_SEQ,
    dtype=torch.bfloat16,
    load_in_4bit=False,
    load_in_8bit=False,
    full_finetuning=False,
    trust_remote_code=True,
    attn_implementation="eager",
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("Base model loaded via Unsloth (bf16, no quantization).")

TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj",
           "in_proj", "out_proj", "up_proj", "down_proj"]
print(f"Attaching LoRA adapter (rank={LORA_RANK}, targets={TARGETS})...")
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=TARGETS,
    lora_alpha=32,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)
model.print_trainable_parameters()


## 2 · Evaluation helpers

Category detection, answer extraction/comparison, and a fixed set of held-out rows used to sanity-check the adapter before submitting.

In [ ]:
# Evaluation helpers:
#   categorize()      - detect the puzzle category from the prompt
#   extract_answer()  - pull the \boxed{} answer out of model text
#   compare_answer()  - exact match for binary/string, 1% relative tolerance for numbers
# Also pick a fixed, seeded set of held-out rows (EVAL_IDS) for the accuracy check.

import re
import math
import random

PROMPT_SUFFIX = ("\nPlease put your final answer inside `\\boxed{}`. "
                 "For example: `\\boxed{your answer}`")


def categorize(prompt: str) -> str:
    """Detect the puzzle category from a keyword in the prompt."""
    if "bit manipulation" in prompt:   return "bit_manipulation"
    if "gravitational" in prompt:      return "gravity"
    if "unit conversion" in prompt:    return "unit_conversion"
    if "encryption" in prompt:         return "cipher"
    if "numeral system" in prompt:     return "numeral"
    if "equations" in prompt:          return "equation"
    return "unknown"


def extract_answer(text: str) -> str:
    matches = re.findall(r"\\boxed\{([^}]*)(?:\}|$)", text)
    if matches:
        non_empty = [m.strip() for m in matches if m.strip()]
        return non_empty[-1] if non_empty else matches[-1].strip()
    return ""


def compare_answer(stored, predicted) -> bool:
    stored = str(stored).strip()
    predicted = str(predicted).strip()
    if re.fullmatch(r"[01]+", stored):
        return predicted.lower() == stored.lower()
    try:
        return math.isclose(float(stored), float(predicted), rel_tol=1e-2, abs_tol=1e-5)
    except Exception:
        return predicted.lower() == stored.lower()


EVAL_PER_CAT = 20
_rng_eval = random.Random(777)
EVAL_IDS = set()
for _cat in ["numeral", "gravity", "unit_conversion", "cipher",
             "bit_manipulation", "equation"]:
    _ids = sorted(r["id"] for r in train.iter_rows(named=True)
                  if categorize(r["prompt"]) == _cat)
    EVAL_IDS.update(_rng_eval.sample(_ids, EVAL_PER_CAT))
print(f"Gate benchmark: {len(EVAL_IDS)} rows ({EVAL_PER_CAT}/category), held out for evaluation.")


## 3 · Build the training set and fine-tune

Clean each teacher trace so its reasoning and final answer agree, format as user/assistant chat, and run one epoch of LoRA SFT with category-balanced batches.

In [ ]:
# Build the supervised fine-tuning set from the teacher chain-of-thought:
#   - answer-consistency cleaning: keep a trace only if it concludes the true answer;
#     add a one-line rounding note for numeric near-misses; drop genuine teacher errors
#   - format as user/assistant chat (thinking enabled) and tokenize
#   - train one epoch with category-stratified batch ordering, then save the adapter.

import pandas as pd, random, time, re, math
from collections import defaultdict
from datasets import Dataset as HFDataset
from trl import SFTTrainer, SFTConfig
from torch.utils.data import DataLoader, Sampler

SEED = 42
NUM_EPOCHS = 1

df = pd.read_csv(COT_CSV)
print(f"Teacher CoT rows: {len(df)}")

before = len(df)
df = df[~df["id"].isin(EVAL_IDS)].reset_index(drop=True)
print(f"Held out {before - len(df)} gate rows; training pool: {len(df)}")

print("\nType distribution:")
print(df["type"].value_counts().sort_index())

train_df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)


def boxed_list(s):
    return re.findall(r"\\boxed\{([^}]*)\}", s)


def build_assistant_content(cot, answer):
    """answer-consistency cleaning (the +score fix over naive boxed-stripping).

    The teacher CoT ends '...The answer in \\boxed{-} is \\boxed{<X>}\n</think>\n\\boxed{<X>}'.
    Stripping ALL \\boxed{} then forcing the rounded dataset answer onto a trace that
    concluded a different number teaches 'reasoning != answer'. Instead: cut at </think>,
    keep the teacher's derived <X>; if <X>!=answer but numerically close add one rounding
    sentence; if truly different (teacher error) drop the row.
    """
    boxes = boxed_list(cot)
    answer_str = str(answer).strip()
    if len(boxes) < 2:
        cleaned = re.sub(r"\\boxed\{[^}]*\}", "", cot).rstrip()
        return cleaned + f"\n</think>\n\\boxed{{{answer_str}}}", True
    cot_final = boxes[-1].strip()
    think_close = cot.rfind("</think>")
    body = cot[:think_close].rstrip() if think_close != -1 else cot.rstrip()
    if cot_final == answer_str:
        rounding_note = ""
    else:
        try:
            if abs(float(cot_final) - float(answer_str)) < 1.0:
                rounding_note = (f"\n\nRounding {cot_final} to match the required "
                                 f"precision gives {answer_str}.")
            else:
                return None, False
        except ValueError:
            return None, False
    assistant_content = body + rounding_note + f"\n</think>\n\\boxed{{{answer_str}}}"
    return assistant_content, True


records = []
record_types = []
n_dropped = 0
for _, row in train_df.iterrows():
    prompt = str(row["prompt"])
    answer = str(row["answer"])
    cot = str(row["generated_cot"])
    if not cot or cot == "nan" or len(cot.strip()) < 5:
        n_dropped += 1
        continue
    assistant_content, ok = build_assistant_content(cot, answer)
    if not ok:
        n_dropped += 1
        continue
    user_content = prompt + PROMPT_SUFFIX
    records.append({"messages": [
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": assistant_content},
    ]})
    record_types.append(str(row["type"]))

dataset = HFDataset.from_list(records)
print(f"\nSFT records: {len(records)} (dropped {n_dropped})")


def formatting_prompts_func(example):
    messages = example["messages"]
    conversations = [messages] if (messages and isinstance(messages[0], dict)) else messages
    texts = []
    for conversation in conversations:
        try:
            text = tokenizer.apply_chat_template(
                conversation, tokenize=False, add_generation_prompt=False, enable_thinking=True)
        except TypeError:
            text = tokenizer.apply_chat_template(
                conversation, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return texts


training_args = SFTConfig(
    output_dir="/kaggle/working/sft_output",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    max_length=8192,
    adam_beta1=0.9,
    adam_beta2=0.95,
    adam_epsilon=1e-8,
    weight_decay=0.0,
    max_grad_norm=1e9,
    logging_steps=10,
    save_strategy="no",
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataloader_num_workers=2,
    remove_unused_columns=False,
    seed=SEED,
    report_to="none",
    packing=False,
)


def build_stratified_index_order(labels, batch_size, seed):
    by_label = defaultdict(list)
    for idx, label in enumerate(labels):
        by_label[label].append(idx)
    rng = random.Random(seed)
    for idx_list in by_label.values():
        rng.shuffle(idx_list)
    n_batches = max(1, math.ceil(len(labels) / batch_size))
    batches = [[] for _ in range(n_batches)]
    batch_order = list(range(n_batches))
    rng.shuffle(batch_order)
    assigned = 0
    for label in sorted(by_label.keys()):
        for idx in by_label[label]:
            batches[batch_order[assigned % n_batches]].append(idx)
            assigned += 1
    return [idx for batch in batches for idx in batch]


class PrecomputedOrderSampler(Sampler):
    def __init__(self, order):
        self.order = list(order)

    def __iter__(self):
        return iter(self.order)

    def __len__(self):
        return len(self.order)


class StratifiedSFTTrainer(SFTTrainer):
    def __init__(self, *args, stratified_order=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.stratified_order = stratified_order

    def get_train_dataloader(self):
        if self.stratified_order is None:
            return super().get_train_dataloader()
        dataloader_kwargs = {
            "batch_size": self.args.per_device_train_batch_size,
            "sampler": PrecomputedOrderSampler(self.stratified_order),
            "collate_fn": self.data_collator,
            "num_workers": self.args.dataloader_num_workers,
            "pin_memory": self.args.dataloader_pin_memory,
            "persistent_workers": self.args.dataloader_persistent_workers,
            "drop_last": self.args.dataloader_drop_last,
        }
        if self.args.dataloader_num_workers > 0:
            dataloader_kwargs["prefetch_factor"] = self.args.dataloader_prefetch_factor
        return DataLoader(self.train_dataset, **dataloader_kwargs)


effective_batch_size = max(1, training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)
stratified_order = build_stratified_index_order(record_types, effective_batch_size, SEED)
print(f"Effective batch size: {effective_batch_size}")
print("Type counts:", dict(sorted(pd.Series(record_types).value_counts().to_dict().items())))

trainer = StratifiedSFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
    formatting_func=formatting_prompts_func,
    stratified_order=stratified_order,
)

print(">>> TRAINING STARTING <<<")
t0 = time.time()
trainer.train()
print(f"Training done in {(time.time() - t0) / 60:.1f} min")

model.save_pretrained(OUTPUT_DIR)
print("Adapter saved to", OUTPUT_DIR)


## 4 · Package the submission

Zip the adapter weights + config into `submission.zip` (done before the accuracy check so a slow eval can never lose the submission).

In [ ]:
# Package the two adapter files (weights + config) into submission.zip.

import os
import json
import subprocess
import zipfile

ADAPTER_CONFIG = os.path.join(OUTPUT_DIR, "adapter_config.json")
ADAPTER_WEIGHTS = os.path.join(OUTPUT_DIR, "adapter_model.safetensors")
SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, "submission.zip")

assert os.path.exists(ADAPTER_CONFIG), "adapter_config.json missing"
assert os.path.exists(ADAPTER_WEIGHTS), "adapter_model.safetensors missing"

with open(ADAPTER_CONFIG) as f:
    cfg = json.load(f)
cfg["base_model_name_or_path"] = BASE_MODEL_NAME
cfg["inference_mode"] = True
cfg["lora_dropout"] = 0.0
with open(ADAPTER_CONFIG, "w") as f:
    json.dump(cfg, f, indent=2)

if os.path.exists(SUBMISSION_ZIP):
    os.remove(SUBMISSION_ZIP)
subprocess.run(["zip", "-j", SUBMISSION_ZIP, ADAPTER_CONFIG, ADAPTER_WEIGHTS], check=True)

_names = sorted(zipfile.ZipFile(SUBMISSION_ZIP).namelist())
assert _names == ["adapter_config.json", "adapter_model.safetensors"], _names
print("Submission packaged before the gate:", SUBMISSION_ZIP, "->", _names)


## 5 · Quick inference test

Run the adapter on the public test rows as a final smoke test.

In [ ]:
# Smoke test: run the fine-tuned model on the public test rows and print the boxed answers.

import os
import torch
import polars as pl
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)
model.eval()
tokenizer.padding_side = "left"


def generate_batch(prompts, max_new_tokens=1024):
    texts = [
        tokenizer.apply_chat_template(
            [{"role": "user", "content": p + PROMPT_SUFFIX}],
            tokenize=False, add_generation_prompt=True, enable_thinking=True,
        )
        for p in prompts
    ]
    enc = tokenizer(texts, return_tensors="pt", padding=True,
                    add_special_tokens=False).to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=max_new_tokens,
                             do_sample=True, temperature=0.6, top_p=0.95,
                             pad_token_id=tokenizer.pad_token_id)
    gen = out[:, enc["input_ids"].shape[1]:]
    return [tokenizer.decode(g, skip_special_tokens=True) for g in gen], None


TEST_CANDIDATES = [
    "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/test.csv",
    "test.csv",
]
TEST_CSV = next((p for p in TEST_CANDIDATES if os.path.exists(p)), TEST_CANDIDATES[0])
test = pl.read_csv(TEST_CSV)
print(f"Loaded {test.height} test rows from {TEST_CSV}")

decoded, _ = generate_batch(test["prompt"].to_list())
for i, txt in enumerate(decoded):
    print(f"{test['id'][i]} -> \\boxed{{{extract_answer(txt)}}}")

## 6 · Clean up the output folder

Reduce the working directory to exactly the submission artifacts.

In [ ]:
# Keep only the submission artefacts (adapter weights + config + submission.zip).

import os
import shutil

SUBMIT_FILES = {"adapter_config.json", "adapter_model.safetensors", "submission.zip"}
KEEP = SUBMIT_FILES
for _f in os.listdir(OUTPUT_DIR):
    if _f in KEEP:
        continue
    _p = os.path.join(OUTPUT_DIR, _f)
    (shutil.rmtree if os.path.isdir(_p) else os.remove)(_p)

for _f in sorted(SUBMIT_FILES):
    assert os.path.exists(os.path.join(OUTPUT_DIR, _f)), f"{_f} missing after cleanup"
print("Final /kaggle/working contents:", sorted(os.listdir(OUTPUT_DIR)))
